<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>

# GP2 × ASAP CRN Interoperability

Links ASAP CRN PMDBS sc/snRNA-seq samples to GP2 participants using the GP2 ID recorded in CRN metadata, and joins the GP2 master key (study, phenotype, genetic data availability) onto each CRN sample.

**Inputs**
- Cleaned CRN metadata from *Assembling CRN Metadata* (one row per sample)
- GP2 Tier 2 release 12 master key (`clinical_data/`)

**Outputs** (`ws_files/GP2xASAPCRN/interoperability/`)
- `gp2_coverage.csv`: how many CRN samples and subjects have a GP2 ID
- `joined_ASAPCRN-GP2_metadata.csv`: CRN metadata with GP2 master columns added (prefixed `gp2_`)
- `gp2_match_summary.csv`: match status counts
- `gp2_unmatched_ids.csv`: CRN GP2 IDs not found in the GP2 master, for follow-up
- `gp2_master_conflicts.csv`: GP2 IDs whose master rows disagree on key fields (only written if there are any)
  
aming:** unprefixed columns come from CRN; `gp2_` columns come from the GP2 master key.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)

WS_ROOT = Path.home() / "workspace"
GP2_RELEASE = "release12"

GP2_MASTER_PATH = (WS_ROOT / "GP2_Tier2" / f"gp2_tier2_eu_{GP2_RELEASE}" / "clinical_data"
                   / f"master_key_{GP2_RELEASE}_final_vwb.csv")

CRN_METADATA_PATH = (WS_ROOT / "ws_files" / "crn_metadata" / "cohort-pmdbs-sc-rnaseq"
                     / "cohort-pmdbs-sc-rnaseq_cleanedmetadata.csv")

OUTPUT_DIR = WS_ROOT / "ws_files" / "GP2xASAPCRN" / "interoperability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load CRN Metadata

Load the cleaned CRN table and normalize its GP2 IDs so they match the GP2 master format: trim spaces, remove sample suffixes like `_s1`, and uppercase.

In [ ]:
crn = pd.read_csv(CRN_METADATA_PATH, low_memory=False)
print(f"CRN: {len(crn):,} rows, {crn.shape[1]:,} columns")

# Strip sample suffixes (_s1, _S2, _s10, ...) so a GP2 ID points to the person
crn["crn_gp2_id_clean"] = (
    crn["GP2_id"].astype("string").str.strip()
    .str.replace(r"_[sS]\d+$", "", regex=True)
    .str.upper()
)

# CRN's own copy of the GP2 phenotype; renamed so it isn't confused with gp2_ columns
crn = crn.rename(columns={"gp2_phenotype": "crn_gp2_phenotype"})

crn.head()

### 2.1 GP2 ID coverage in CRN
How many CRN samples and subjects have a GP2 ID. A sample can appear in more than one dataset, so samples are counted by dataset + sample ID.

In [ ]:
has_gp2 = crn["crn_gp2_id_clean"].notna()
sample_key = crn["ASAP_dataset_id"] + "|" + crn["ASAP_sample_id"]

gp2_coverage = pd.DataFrame({
    "level": ["samples", "subjects"],
    "with_GP2_id": [sample_key[has_gp2].nunique(),
                    crn.loc[has_gp2, "ASAP_subject_id"].nunique()],
    "total": [sample_key.nunique(), crn["ASAP_subject_id"].nunique()],
})
gp2_coverage["pct"] = (100 * gp2_coverage["with_GP2_id"] / gp2_coverage["total"]).round(1)

gp2_coverage.to_csv(OUTPUT_DIR / "gp2_coverage.csv", index=False)
gp2_coverage

### 2.2 Map CRN brain banks to GP2 studies
A CRN-side guess at which GP2 study each sample belongs to, based on its brain bank. It's used only as a cross-check against the GP2 master's study in Section 5; the master is the authoritative source.

In [ ]:
BIOBANK_TO_GP2_STUDY = {
    "Banner Sun Health Research Institute": "BBDP",
    "Banner_Sun_Health_USA": "BBDP",
    "QSBB_UK": "MDGAP-QSBB",
}
crn["crn_expected_gp2_study"] = crn["biobank_name"].map(BIOBANK_TO_GP2_STUDY)

crn["biobank_name"].value_counts(dropna=False).to_frame("samples")

## 3. Load the GP2 Master Key

Rename the columns used in this notebook to clear names. All GP2 columns get a `gp2_` prefix in Section 3.2.

In [ ]:
gp2 = pd.read_csv(GP2_MASTER_PATH, low_memory=False)

gp2 = gp2.rename(columns={
    "GP2ID": "GP2_ID",
    "study": "Study",
    "GP2_PHENO": "Phenotype",
    "nba": "NBA",
    "wgs": "WGS",
    "nba_prune_reason": "NBA_Prune_Reason",
    "wgs_prune_reason": "WGS_Prune_Reason",
    "sample_id": "Source_Sample_ID",
    "race_for_qc": "Race",
    "biological_sex_for_qc": "Sex",
    "age_at_death": "Age_At_Death",
})
gp2["GP2_ID"] = gp2["GP2_ID"].astype("string").str.strip().str.upper()

print(f"GP2 master: {len(gp2):,} rows, {gp2['GP2_ID'].nunique():,} unique GP2 IDs")
gp2.head()

### 3.1 Collapse to one row per GP2 ID
Some participants have more than one master row (for example, separate NBA and WGS records). Joining on these would duplicate CRN samples, so they're collapsed first:
- `NBA` and `WGS` are 0/1 availability flags, so a participant has the data if **any** of their rows is 1.
- Every other column keeps the first non-empty value. Participants whose rows disagree on key fields are listed below.

In [ ]:
# NBA / WGS should only be 0 or 1
for col in ["NBA", "WGS"]:
    assert set(gp2[col].dropna().unique()) <= {0, 1}, f"Unexpected values in {col}"
    print(f"{col}: {gp2[col].isna().sum():,} empty values")

gp2["Has_NBA"] = gp2["NBA"].eq(1)
gp2["Has_WGS"] = gp2["WGS"].eq(1)

# Flag participants whose rows disagree on fields that matter for analysis
check_cols = ["Phenotype", "Sex", "Race", "Age_At_Death"]
n_values = gp2.groupby("GP2_ID")[check_cols].nunique(dropna=True)
gp2_conflicts = n_values[(n_values > 1).any(axis=1)]
print(f"GP2 IDs with conflicting values across rows: {len(gp2_conflicts):,}")
if len(gp2_conflicts):
    gp2_conflicts.to_csv(OUTPUT_DIR / "gp2_master_conflicts.csv")
    display(gp2_conflicts.head())

# Collapse: any() for availability flags, first non-empty value for everything else
agg = {col: "first" for col in gp2.columns if col != "GP2_ID"}
agg.update({"Has_NBA": "any", "Has_WGS": "any"})
gp2_one = gp2.groupby("GP2_ID", as_index=False).agg(agg)

print(f"Collapsed: {len(gp2):,} rows -> {len(gp2_one):,} (one per GP2 ID)")

### 3.2 Prefix GP2 columns
Every GP2 column except the join key gets a `gp2_` prefix, so each field shows where it came from and no columns collide during the merge.

In [ ]:
gp2_one = gp2_one.rename(columns={c: f"gp2_{c}" for c in gp2_one.columns if c != "GP2_ID"})

overlap = (set(crn.columns) & set(gp2_one.columns)) - {"GP2_ID"}
assert not overlap, f"Columns in both tables: {sorted(overlap)}"
print("No overlapping columns.")

## 4. Join GP2 onto CRN

A left join keeps every CRN sample. `validate="m:1"` confirms each CRN sample matches at most one GP2 participant, and the row count check confirms no samples were added or lost.

Each sample gets a match status:
| Status | Meaning |
|---|---|
| `matched` | CRN GP2 ID found in the GP2 master |
| `not_in_gp2_master` | CRN has a GP2 ID, but it isn't in this release's master key |
| `no_gp2_id` | CRN has no GP2 ID for this sample |

In [ ]:
merged = crn.merge(
    gp2_one,
    left_on="crn_gp2_id_clean", right_on="GP2_ID",
    how="left", validate="m:1",
)
assert len(merged) == len(crn), "Merge changed the number of CRN rows."

merged["gp2_match_status"] = np.select(
    [merged["crn_gp2_id_clean"].isna(), merged["GP2_ID"].isna()],
    ["no_gp2_id", "not_in_gp2_master"],
    default="matched",
)

match_summary = merged["gp2_match_status"].value_counts().rename_axis("status").reset_index(name="samples")
match_summary

## 5. Checks

### 5.1 IDs not found in the GP2 master
These CRN GP2 IDs don't appear in this release. Possible reasons include a typo in the CRN submission, a participant who isn't in this GP2 release yet, or a different ID format.

In [ ]:
unmatched = (
    merged.loc[merged["gp2_match_status"] == "not_in_gp2_master",
               ["GP2_id", "crn_gp2_id_clean", "ASAP_subject_id", "ASAP_dataset_id", "biobank_name"]]
    .drop_duplicates()
)
print(f"Unmatched GP2 IDs: {unmatched['crn_gp2_id_clean'].nunique():,}")
unmatched.to_csv(OUTPUT_DIR / "gp2_unmatched_ids.csv", index=False)
unmatched.head(20)

### 5.2 Brain bank vs GP2 study
For matched samples, compare the study expected from the CRN brain bank with the study in the GP2 master. Rows where they disagree may point to a wrong GP2 ID.

In [ ]:
matched = merged[merged["gp2_match_status"] == "matched"]
pd.crosstab(
    matched["crn_expected_gp2_study"].fillna("(no mapping)"),
    matched["gp2_Study"].fillna("(empty)"),
    rownames=["Expected (CRN brain bank)"], colnames=["GP2 master study"],
)

### 5.3 Genetic data available for matched participants

In [ ]:
matched_people = matched.drop_duplicates("GP2_ID")
pd.Series({
    "Matched GP2 participants": len(matched_people),
    "With NBA (array) data": int(matched_people["gp2_Has_NBA"].sum()),
    "With WGS data": int(matched_people["gp2_Has_WGS"].sum()),
}).to_frame("count")

## 6. Save Outputs

In [ ]:
merged.to_csv(OUTPUT_DIR / "joined_ASAPCRN-GP2_metadata.csv", index=False)
match_summary.to_csv(OUTPUT_DIR / "gp2_match_summary.csv", index=False)

print("Saved to", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")